In [ ]:
!pip install -q torch-fidelity open-clip-torch lpips scikit-image tqdm

import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/liya_diplomCC'
    AI_TOOLKIT = '/content/ai-toolkit'
    sys.path.insert(0, DRIVE_ROOT)
    sys.path.insert(0, AI_TOOLKIT)
except ModuleNotFoundError:
    # Local run: find project root by looking for scripts/ directory
    _here = Path().resolve()
    DRIVE_ROOT = str(_here if (_here / 'scripts').exists() else _here.parent)
    AI_TOOLKIT = str(Path(DRIVE_ROOT).parent / 'ai-toolkit')
    sys.path.insert(0, DRIVE_ROOT)
    sys.path.insert(0, AI_TOOLKIT)

print(f"DRIVE_ROOT: {DRIVE_ROOT}")

In [ ]:
import json, shutil
from pathlib import Path

real_dir = f'{DRIVE_ROOT}/data/png_512_test'
Path(real_dir).mkdir(parents=True, exist_ok=True)

with open(f'{DRIVE_ROOT}/data/test_500.jsonl') as f:
    test_pairs = [json.loads(l) for l in f]

for item in test_pairs:
    shutil.copy(item['png_path'], real_dir)

print(f"Copied {len(test_pairs)} real images to {real_dir}")

In [ ]:
from scripts.compute_metrics import compute_fid, compute_clip_score
from pathlib import Path
import json

captions = [p['caption'] for p in test_pairs]

EXPERIMENTS = {
    "SD1.5 Baseline":  f'{DRIVE_ROOT}/results/experiments/sd15_baseline',
    "SDXL LoRA r4":    f'{DRIVE_ROOT}/results/experiments/sdxl_r4_samples',
    "SDXL LoRA r8":    f'{DRIVE_ROOT}/results/experiments/sdxl_r8_samples',
    "SDXL LoRA r16":   f'{DRIVE_ROOT}/results/experiments/sdxl_r16_samples',
    "SDXL LoRA r32":   f'{DRIVE_ROOT}/results/experiments/sdxl_r32_samples',
    "SDXL r16 s500":   f'{DRIVE_ROOT}/results/experiments/sdxl_r16_s500_samples',
    "SDXL r16 s1000":  f'{DRIVE_ROOT}/results/experiments/sdxl_r16_s1000_samples',
    "SDXL r16 s4000":  f'{DRIVE_ROOT}/results/experiments/sdxl_r16_s4000_samples',
    "FLUX LoRA r16":   f'{DRIVE_ROOT}/results/experiments/flux_r16_samples',
    "Recraft v3":      f'{DRIVE_ROOT}/results/experiments/exp4_recraft',
}

results = {}
for name, fake_dir in EXPERIMENTS.items():
    fake_dir_path = Path(fake_dir)
    if not fake_dir_path.exists():
        print(f"SKIP {name}: directory not found")
        continue
    fake_imgs = sorted(fake_dir_path.glob("*.png"))
    if not fake_imgs:
        print(f"SKIP {name}: no images found")
        continue
    caps = captions[:len(fake_imgs)]
    fid  = compute_fid(real_dir, fake_dir)
    clip = compute_clip_score([str(p) for p in fake_imgs], caps)
    results[name] = {"fid": fid, "clip_score": clip}
    print(f"{name:<22} FID={fid:6.2f}  CLIP={clip:.4f}")

with open(f'{DRIVE_ROOT}/results/metrics/all_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nSaved metrics for {len(results)} experiments")

In [ ]:
from scripts.scale_test import scale_test_batch
import json

scale_results = {}
for name, fake_dir in EXPERIMENTS.items():
    if not Path(fake_dir).exists() or not list(Path(fake_dir).glob("*.png")):
        continue
    try:
        scale_results[name] = scale_test_batch(fake_dir)
        s = scale_results[name]
        print(f"{name:<22} SSIM@16={s[16]:.3f}  @64={s[64]:.3f}  @512={s[512]:.3f}")
    except ValueError as e:
        print(f"SKIP {name}: {e}")

with open(f'{DRIVE_ROOT}/results/metrics/scale_test.json', 'w') as f:
    json.dump(scale_results, f, indent=2)

In [ ]:
from scripts.caption_llava import load_model
from pathlib import Path
from tqdm import tqdm
import json

processor, vlm_model, device = load_model()

VLM_QUESTIONS = [
    "Is this image a professional logo? Rate 0-5 where 5=definitely a logo. Reply with a single digit only.",
    "Rate the aesthetic quality of this logo design 0-5 where 5=excellent. Reply with a single digit only.",
]

def vlm_score(img_path: str, question: str) -> float:
    from PIL import Image as PILImage
    prompt = f"USER: <image>\n{question}\nASSISTANT:"
    image = PILImage.open(img_path).convert("RGB")
    inputs = processor(prompt, image, return_tensors="pt").to(device)
    import torch
    with torch.no_grad():
        output = vlm_model.generate(**inputs, max_new_tokens=5, do_sample=False)
    raw = processor.decode(output[0], skip_special_tokens=True).split("ASSISTANT:")[-1].strip()
    try:
        return min(max(float(raw[0]), 0.0), 5.0)
    except (ValueError, IndexError):
        return 0.0

vlm_results = {}
for name, fake_dir in list(EXPERIMENTS.items()):
    if not Path(fake_dir).exists():
        continue
    images = sorted(Path(fake_dir).glob("*.png"))[:20]
    if not images:
        continue
    logo_scores, quality_scores = [], []
    for img_path in tqdm(images, desc=name):
        logo_scores.append(vlm_score(str(img_path), VLM_QUESTIONS[0]))
        quality_scores.append(vlm_score(str(img_path), VLM_QUESTIONS[1]))
    vlm_results[name] = {
        "logo_score":    sum(logo_scores) / len(logo_scores),
        "quality_score": sum(quality_scores) / len(quality_scores),
    }
    print(f"{name:<22} logo={vlm_results[name]['logo_score']:.2f}  "
          f"quality={vlm_results[name]['quality_score']:.2f}")

with open(f'{DRIVE_ROOT}/results/metrics/vlm_scores.json', 'w') as f:
    json.dump(vlm_results, f, indent=2)

In [ ]:
import matplotlib.pyplot as plt
import json

with open(f'{DRIVE_ROOT}/results/metrics/all_metrics.json') as f:
    metrics = json.load(f)

if not metrics:
    print("No metrics to plot — run Cell 3 first")
else:
    names = list(metrics.keys())
    fids  = [metrics[n]['fid'] for n in names]
    clips = [metrics[n]['clip_score'] for n in names]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    ax1.barh(names, fids, color='steelblue')
    ax1.set_xlabel("FID (lower is better)")
    ax1.set_title("FID by Model/Config")
    sd15_fid = metrics.get("SD1.5 Baseline", {}).get("fid")
    if sd15_fid is not None:
        ax1.axvline(sd15_fid, color='red', linestyle='--', alpha=0.5, label='SD1.5 baseline')
        ax1.legend()

    ax2.barh(names, clips, color='darkorange')
    ax2.set_xlabel("CLIP Score (higher is better)")
    ax2.set_title("CLIP Score by Model/Config")

    plt.tight_layout()
    plt.savefig(f'{DRIVE_ROOT}/results/metrics/metrics_comparison.png', dpi=150)
    plt.show()

    print(f"\n{'Model':<25} {'FID':>8} {'CLIP Score':>12}")
    print("-" * 47)
    for name in names:
        print(f"{name:<25} {metrics[name]['fid']:>8.2f} {metrics[name]['clip_score']:>12.4f}")